In [8]:
import pandas as pd
import gc

## Leitura dos Dados e Remoção do Cabeçalho

A planilha do IBGE possui as informações estruturadas a partir da linha 9 (índice 8). Lemos o arquivo pulando as 8 primeiras linhas de cabeçalhos descritivos.

In [9]:
# Lemos o arquivo pulando as 8 primeiras linhas de cabeçalho do Excel
df_censo = pd.read_excel('censo_2022/dados_raciais_2022_MU.xlsx', skiprows=8, header=None)

print(f"Tamanho inicial do dataset: {df_censo.shape}")
display(df_censo.head())

Tamanho inicial do dataset: (5572, 16)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Absoluto,Percentual
1,Alta Floresta D'Oeste,1100015.0,21 494,7 590,1 297,46,11 845,715,1,35.313823,6.034523,0.214023,55.110966,3.326664,749,3.484693
2,Ariquemes,1100023.0,96 833,31 263,9 619,317,55 534,97,3,32.286481,9.933905,0.327378,57.352060,0.100176,98,0.101205
3,Cabixi,1100031.0,5 351,2 136,345,12,2 856,2,0,39.917772,6.447393,0.224257,53.373201,0.037376,10,0.186881
4,Cacoal,1100049.0,86 887,32 304,6 704,365,45 836,1 675,3,37.180609,7.716035,0.420100,52.755398,1.927858,1688,1.942753


## Seleção e Renomeação das Colunas

Com base na inspeção da estrutura do arquivo do IBGE, identificamos as colunas de interesse:
- Índice 0: Nome do Município
- Índice 1: Código do Município (Cód.)
- Índice 2: População Total (Absoluto)
- Índice 3: População Branca (Absoluto)
- Índice 4: População Preta (Absoluto)
- Índice 5: População Amarela (Absoluto)
- Índice 6: População Parda (Absoluto)
- Índice 7: População Indígena (Absoluto)

Selecionamos apenas essas colunas e as renomeamos para o padrão adotado no projeto.

In [10]:
# Selecionamos as colunas pelos índices numéricos
colunas_interesse = [0, 1, 2, 3, 4, 5, 6, 7]
df_censo = df_censo[colunas_interesse].copy()

# Dicionário de renomeação para padronizar as variáveis
colunas_map = {
    0: 'city_name',
    1: 'city_code',
    2: 'population_total',
    3: 'population_white',
    4: 'population_black',
    5: 'population_yellow',
    6: 'population_brown',
    7: 'population_indigenous'
}

# Aplicamos o novo nome às colunas
df_censo = df_censo.rename(columns=colunas_map)

display(df_censo.head())

,city_name,city_code,population_total,population_white,population_black,population_yellow,population_brown,population_indigenous
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Alta Floresta D'Oeste,1100015.0,21 494,7 590,1 297,46,11 845,715
2,Ariquemes,1100023.0,96 833,31 263,9 619,317,55 534,97
3,Cabixi,1100031.0,5 351,2 136,345,12,2 856,2
4,Cacoal,1100049.0,86 887,32 304,6 704,365,45 836,1 675


## 4. Limpeza de Rodapés e Conversão de Tipos

Os arquivos do IBGE costumam trazer informações textuais (fontes, notas técnicas) nas últimas linhas, o que converte as colunas numéricas para o tipo `object` (string). Vamos limpar esses rodapés forçando a conversão da coluna `city_code` para numérico. Linhas de rodapé se tornarão nulas (`NaN`) e poderão ser removidas.

In [11]:
# Forçamos a conversão da coluna 'city_code' para numérico.
# Textos de rodapé do IBGE retornarão NaN.
df_censo['city_code'] = pd.to_numeric(df_censo['city_code'], errors='coerce')

# Removemos apenas as linhas onde o 'city_code' é nulo (rodapés reais)
df_censo = df_censo.dropna(subset=['city_code']).copy()

# Colunas populacionais que serão tratadas
colunas_inteiras = [
    'city_code', 'population_total', 'population_white', 
    'population_black', 'population_yellow', 'population_brown', 'population_indigenous'
]

for col in colunas_inteiras:
    # 1. Converte para string, remove espaços e troca hifens/traços do IBGE por zero
    df_censo[col] = df_censo[col].astype(str).str.replace(' ', '').str.replace('-', '0')
    
    # 2. Converte para numérico (qualquer texto estranho remanescente vira NaN)
    df_censo[col] = pd.to_numeric(df_censo[col], errors='coerce')
    
    # 3. Preenche os NaNs resultantes (ausência de dados) com 0
    df_censo[col] = df_censo[col].fillna(0)
    
    # 4. Converte para inteiro
    df_censo[col] = df_censo[col].astype(int)

print(f"Total de municípios após a limpeza: {len(df_censo)}")

Total de municípios após a limpeza: 5570


In [12]:
df_censo

,city_name,city_code,population_total,population_white,population_black,population_yellow,population_brown,population_indigenous
1,Alta Floresta D'Oeste,1100015,21494,7590,1297,46,11845,715
2,Ariquemes,1100023,96833,31263,9619,317,55534,97
3,Cabixi,1100031,5351,2136,345,12,2856,2
4,Cacoal,1100049,86887,32304,6704,365,45836,1675
5,Cerejeiras,1100056,15890,6213,1025,59,8580,13
...,...,...,...,...,...,...,...,...
5566,Vianópolis,5222005,14956,6328,954,14,7645,15
5567,Vicentinópolis,5222054,8768,3349,715,10,4683,11
5568,Vila Boa,5222203,4215,899,861,3,2444,8
5569,Vila Propício,5222302,5815,1407,533,24,3844,5


## Exportação do Dataset Preparado

Com o arquivo limpo, estruturado e tipado, nós o exportamos no formato CSV para que seja integrado rapidamente no notebook principal de EDA.

In [13]:
# Exportação para CSV
df_censo.to_csv('censo_2022/censo_2022.csv', index=False, sep=';', encoding='utf-8')